# OFFBALL — Regresión del valor de mercado

**Objetivo:** estimar el valor de mercado de un futbolista (en euros) usando su edad, posición y estadísticas de rendimiento.

Este notebook prepara los datos, construye `df2_regression`, compara modelos con validación temporal y evalúa el modelo elegido sobre valoraciones futuras.

- No usamos PySpark.
- Los CSV limpios originales no se modifican.
- Solo usamos estadísticas de temporadas terminadas **antes** de cada valoración.
- **Validación:** `TimeSeriesSplit` sobre los años hasta 2023. Comparamos los modelos en tres periodos temporales.
- **Test final:** 2024 en adelante. Se reserva para la evaluación final y no se usa para elegir el modelo.
- Un mismo jugador puede aparecer en años distintos; evaluamos predicciones futuras de jugadores que pueden ser conocidos.
- MLflow no lo instalamos en Colab. Si está disponible, registra los experimentos; si no, el notebook guarda los archivos localmente en Drive.
- Las variables de clustering/GMM se incorporarán más adelante como un experimento separado.

Ejecuta las celdas en orden. Cada sección explica qué hacemos y por qué.

## Cómo leer las explicaciones añadidas

Además de indicar qué hace cada bloque, estas notas explican **por qué se ha elegido ese método**, qué problema evita y cómo interpretar el resultado. El código original y sus resultados se mantienen tal como estaban.

La tarea es de **regresión** porque queremos predecir una cantidad continua: el valor de mercado en euros. No estamos clasificando a los jugadores en grupos.


## 1. Cargar datos y preparar rutas

Montamos Google Drive, definimos las carpetas del proyecto y cargamos solo las columnas que necesitamos para reducir el uso de memoria.

In [ ]:
# ==========================================================
# 1. CARGA DE DATOS, RUTAS Y CACHÉ
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Carpeta principal del proyecto en Drive
OFFBALL_DIR = Path('/content/drive/MyDrive/PROYECTO-IGOR-CARMELO-BYRON/OFFBALL')
CLEAN_DIR = OFFBALL_DIR / 'dataset_limpio'
OUTPUT_DIR = OFFBALL_DIR / 'Machine_Learning' / 'DF2_Regresion'
MODELS_DIR = OUTPUT_DIR / 'modelos'
MLFLOW_DIR = OUTPUT_DIR / 'mlruns'

for carpeta in [OUTPUT_DIR, MODELS_DIR, MLFLOW_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)


archivos_necesarios = ['players.csv', 'appearances.csv', 'games.csv', 'player_valuations.csv']
rutas_fuente = [CLEAN_DIR / archivo for archivo in archivos_necesarios]
for ruta in rutas_fuente:
    if not ruta.exists():
        raise FileNotFoundError(f'No se encuentra el archivo: {ruta}')

# El caché se invalida automáticamente si cambia el tamaño o fecha de alguno de los CSV limpios.
CACHE_DF2 = OUTPUT_DIR / 'cache_df2_regression.csv'
CACHE_AUDITORIA = OUTPUT_DIR / 'cache_auditoria_filtros.csv'
CACHE_COBERTURA = OUTPUT_DIR / 'cache_cobertura_estadisticas.csv'
CACHE_METADATA = OUTPUT_DIR / 'cache_metadata.json'
FIRMA_FUENTES = {
    ruta.name: {'size': ruta.stat().st_size, 'mtime_ns': ruta.stat().st_mtime_ns}
    for ruta in rutas_fuente
}
CACHE_CARGADO = False

if all(ruta.exists() for ruta in [CACHE_DF2, CACHE_AUDITORIA, CACHE_COBERTURA, CACHE_METADATA]):
    try:
        with open(CACHE_METADATA, 'r', encoding='utf-8') as f:
            metadata_cache = json.load(f)
        if metadata_cache.get('firma_fuentes') == FIRMA_FUENTES:
            df2_regression = pd.read_csv(CACHE_DF2, parse_dates=['valuation_date', 'latest_stats_end'])
            df_auditoria_filtros = pd.read_csv(CACHE_AUDITORIA)
            df_cobertura_anual = pd.read_csv(CACHE_COBERTURA)
            CACHE_CARGADO = True
            print('CACHÉ VÁLIDO: dataset de regresión cargado desde Drive.')
            print('Se saltan las uniones y agregaciones grandes de las secciones 2–7.')
        else:
            print('Caché antiguo: los CSV limpios han cambiado. Se recalculará el dataset.')
    except Exception as error:
        print('No se pudo leer el caché; se recalculará:', type(error).__name__)

if not CACHE_CARGADO:
    print('Primera ejecución o caché inválido: cargando CSV limpios y preparando datos...')
    df_players = pd.read_csv(
        CLEAN_DIR / 'players.csv',
        usecols=['player_id', 'date_of_birth', 'position', 'sub_position']
    )
    df_appearances = pd.read_csv(
        CLEAN_DIR / 'appearances.csv',
        usecols=['appearance_id', 'game_id', 'player_id', 'competition_id', 'minutes_played', 'goals', 'assists']
    )
    df_games = pd.read_csv(
        CLEAN_DIR / 'games.csv',
        usecols=['game_id', 'competition_id', 'season', 'date'],
        parse_dates=['date']
    )
    df_valuations = pd.read_csv(
        CLEAN_DIR / 'player_valuations.csv',
        usecols=['player_id', 'date', 'market_value_in_eur'],
        parse_dates=['date']
    )
    df_valuations['market_value_in_eur'] = pd.to_numeric(
        df_valuations['market_value_in_eur'], errors='coerce'
    )
    resumen_carga = pd.DataFrame([
        {'Dataset': 'players', 'Filas': len(df_players), 'Columnas cargadas': df_players.shape[1]},
        {'Dataset': 'appearances', 'Filas': len(df_appearances), 'Columnas cargadas': df_appearances.shape[1]},
        {'Dataset': 'games', 'Filas': len(df_games), 'Columnas cargadas': df_games.shape[1]},
        {'Dataset': 'player_valuations', 'Filas': len(df_valuations), 'Columnas cargadas': df_valuations.shape[1]},
    ])
    print(resumen_carga.to_string(index=False))
    print('Fechas nulas en valoraciones:', df_valuations['date'].isna().sum())
    print('Valores de mercado nulos:', df_valuations['market_value_in_eur'].isna().sum())
else:
    # Variables de preparación no necesarias en el modo caché.
    print('Filas de DF2 para regresión:', len(df2_regression))
    print('Jugadores:', df2_regression['player_id'].nunique())


### Por qué cargamos estos archivos y usamos una caché

Cada CSV aporta una parte necesaria del problema: `players.csv` contiene edad de nacimiento y posición; `appearances.csv`, minutos, goles y asistencias; `games.csv`, fecha y temporada de cada partido; y `player_valuations.csv`, el valor que queremos predecir. Cargamos solo las columnas necesarias para no ocupar memoria con información que este modelo no utiliza.

En esta ejecución aparece **“CACHÉ VÁLIDO”**: se reutilizó el dataset preparado anteriormente (`df2_regression`), con **291.962 valoraciones y 26.892 jugadores**, en lugar de repetir todas las uniones y agregaciones. Esto ahorra tiempo; la parte de modelos, validación y guardado sí continúa ejecutándose.

**Importante sobre la caché:** el control comprueba el tamaño y la fecha de modificación de los CSV limpios, pero no detecta cambios hechos solamente en el código de preparación (secciones 2–7). Si modificamos esa lógica y queremos reconstruir el dataset, debemos invalidar o borrar manualmente los archivos `cache_` antes de ejecutar. De lo contrario, podríamos seguir usando el `df2_regression` anterior.

El notebook trabaja con pandas en Colab; esto no significa que se haya usado PySpark. La preparación del modelo se mantiene separada del almacenamiento Big Data del proyecto.


**Modo de ejecución rápida:** la primera ejecución prepara los datos desde los CSV limpios y crea un caché en Drive. Si los CSV no han cambiado, las siguientes ejecuciones reutilizan `df2_regression` y saltan las secciones de preparación pesada (2–7), manteniendo la parte de modelos, evaluación y artefactos.

## 2. Comprobar claves y duplicados

Revisamos las claves antes de hacer uniones. Si una clave que debería ser única tiene duplicados, detenemos el notebook para no multiplicar filas accidentalmente.

### Por qué comprobamos las claves antes de unir tablas

Antes de combinar archivos comprobamos que los identificadores que deberían ser únicos realmente lo sean. Por ejemplo, una aparición repetida podría contar dos veces los minutos o los goles; un partido duplicado podría multiplicar apariciones durante la unión; y una valoración duplicada para el mismo jugador y fecha podría hacer que una misma observación pesara más de una vez.

Los `assert` hacen que el notebook se detenga si encuentra esos duplicados, en vez de continuar con un dataset incorrecto sin avisar. También contamos los valores nulos para conocer qué información falta y decidir después cómo tratarla.


In [ ]:
# Esta sección solo se ejecuta si no hay caché válido; evita repetir cálculos pesados.
if not CACHE_CARGADO:
    # ==========================================================
    # 2. CONTROL DE CLAVES Y DUPLICADOS
    # ==========================================================

    duplicados_players = df_players.duplicated('player_id').sum()
    duplicados_appearances = df_appearances.duplicated('appearance_id').sum()
    duplicados_games = df_games.duplicated('game_id').sum()
    duplicados_valuations = df_valuations.duplicated(['player_id', 'date']).sum()

    print('Duplicados player_id en players:', duplicados_players)
    print('Duplicados appearance_id en appearances:', duplicados_appearances)
    print('Duplicados game_id en games:', duplicados_games)
    print('Duplicados jugador-fecha en valoraciones:', duplicados_valuations)

    # Duplicados en estas claves pueden provocar multiplicación de filas en los merges
    assert duplicados_players == 0, 'Hay player_id duplicados en players.csv'
    assert duplicados_appearances == 0, 'Hay appearance_id duplicados en appearances.csv'
    assert duplicados_games == 0, 'Hay game_id duplicados en games.csv'
    assert duplicados_valuations == 0, 'Hay valoraciones duplicadas por jugador y fecha'

    # Conteo de nulos en las columnas cargadas
    nulos_principales = pd.DataFrame({
        'Dataset': ['players', 'appearances', 'games', 'player_valuations'],
        'Nulos en columnas cargadas': [
            df_players.isna().sum().sum(),
            df_appearances.isna().sum().sum(),
            df_games.isna().sum().sum(),
            df_valuations.isna().sum().sum()
        ]
    })
    print('\nNulos totales en las columnas cargadas:')
    print(nulos_principales.to_string(index=False))

## 3. Exploración inicial del valor de mercado

Miramos la evolución anual y la distribución del objetivo. Como los valores son muy desiguales, usamos escala logarítmica en el histograma. El objetivo se mantiene en euros; las métricas del modelo se mostrarán en millones de euros.

### Por qué exploramos el valor antes de entrenar

Primero miramos cómo se comporta la variable que queremos predecir. En fútbol hay muchos jugadores con valores relativamente bajos y unos pocos con valores muy altos. Esos valores extremos pueden elevar mucho la **media**; por eso también mostramos la **mediana**, que representa mejor el centro cuando la distribución está muy desequilibrada.

El gráfico usa escala logarítmica para que se puedan distinguir a la vez los valores pequeños y los grandes. Es una decisión de visualización: **no transforma el objetivo que aprende el modelo**. Así mantenemos las predicciones y los errores en euros reales.


In [ ]:
# Esta sección solo se ejecuta si no hay caché válido; evita repetir cálculos pesados.
if not CACHE_CARGADO:
    # ==========================================================
    # 3. EXPLORACIÓN DEL OBJETIVO
    # ==========================================================

    # Resumen por año: número de valoraciones, jugadores, mediana y media
    df_val_temporal = df_valuations.dropna(subset=['date']).copy()
    df_val_temporal['valuation_year'] = df_val_temporal['date'].dt.year

    df_resumen_anual = (
        df_val_temporal.groupby('valuation_year')
        .agg(
            valoraciones=('player_id', 'size'),
            jugadores=('player_id', 'nunique'),
            mediana_eur=('market_value_in_eur', 'median'),
            media_eur=('market_value_in_eur', 'mean')
        )
        .reset_index()
    )
    df_resumen_anual['mediana_millones'] = df_resumen_anual['mediana_eur'] / 1_000_000
    df_resumen_anual['media_millones'] = df_resumen_anual['media_eur'] / 1_000_000

    print('Período de valoraciones:', df_val_temporal['date'].min(), 'a', df_val_temporal['date'].max())
    print(df_resumen_anual[['valuation_year', 'valoraciones', 'jugadores', 'mediana_millones', 'media_millones']].round(2).to_string(index=False))

    # Mediana anual: es menos sensible a valores extremos que la media
    plt.figure(figsize=(11, 4))
    plt.plot(df_resumen_anual['valuation_year'], df_resumen_anual['mediana_millones'], marker='o')
    plt.title('Mediana del valor de mercado por año')
    plt.xlabel('Año')
    plt.ylabel('Millones de euros')
    plt.xticks(df_resumen_anual['valuation_year'], rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Histograma de valores positivos en escala logarítmica
    valores_positivos = df_valuations.loc[
        df_valuations['market_value_in_eur'] > 0, 'market_value_in_eur'
    ] / 1_000_000
    print('\nValoraciones totales:', df_valuations['market_value_in_eur'].count())
    print('Valoraciones positivas:', len(valores_positivos))
    print('Mediana (M€):', round(valores_positivos.median(), 3))
    print('Media (M€):', round(valores_positivos.mean(), 3))
    print('Nulos:', df_valuations['market_value_in_eur'].isna().sum())
    print('Ceros:', (df_valuations['market_value_in_eur'] == 0).sum())
    print('Negativos:', (df_valuations['market_value_in_eur'] < 0).sum())

    if len(valores_positivos) > 0:
        bins = np.logspace(np.log10(valores_positivos.min()), np.log10(valores_positivos.max()), 45)
        plt.figure(figsize=(10, 4))
        plt.hist(valores_positivos, bins=bins)
        plt.xscale('log')
        plt.title('Distribución del valor de mercado')
        plt.xlabel('Millones de euros (escala logarítmica)')
        plt.ylabel('Valoraciones')
        plt.grid(axis='y')
        plt.tight_layout()
        plt.show()
    else:
        print('No hay valores positivos para representar.')

**Decisión inicial:** usamos el valor en euros original y no aplicamos ajuste de inflación. `valuation_year` se probará como variable temporal, pero no equivale a una corrección de inflación. Transformar el objetivo a logaritmos queda para un experimento posterior.

### Por qué mantenemos el objetivo en euros nominales

El objetivo sigue siendo `market_value_in_eur` en euros originales. Así una predicción y su error se pueden explicar directamente en euros o millones de euros, sin tener que deshacer una transformación matemática.

No aplicamos un ajuste por inflación o por evolución general del mercado. Por tanto, 1 millón de euros de 2015 y 1 millón de euros de 2026 se tratan como la misma cantidad nominal. Esto simplifica la primera línea base, pero es una **limitación temporal** que debe recordarse al interpretar los resultados.

Probamos `valuation_year` como posible variable de entrada para que el modelo pueda reconocer diferencias entre épocas. Incluir el año no corrige automáticamente la inflación: solo ofrece al modelo el año de la valoración como información adicional. El histograma logarítmico tampoco cambia los euros que usa el entrenamiento.


## 4. Calendario de temporadas

Calculamos el inicio y final real de cada competición-temporada a partir de las fechas de los partidos. Usaremos este calendario para no añadir estadísticas de temporadas que todavía no habían terminado cuando se valoró al jugador.

### Por qué calculamos el calendario con las fechas de los partidos

No fijamos una fecha universal de final de temporada porque las competiciones pueden tener calendarios distintos. Agrupamos los partidos por competición y temporada y usamos la primera y la última fecha que aparece en los datos.

La fecha final sirve después para saber si una temporada ya había terminado cuando se hizo una valoración. Esto evita utilizar estadísticas de una temporada que todavía estaba en curso. La fecha calculada depende de que `games.csv` contenga correctamente los partidos de esa competición y temporada.


In [ ]:
# Esta sección solo se ejecuta si no hay caché válido; evita repetir cálculos pesados.
if not CACHE_CARGADO:
    # ==========================================================
    # 4. CALENDARIO COMPETICIÓN-TEMPORADA
    # ==========================================================

    df_season_calendar = (
        df_games.dropna(subset=['date', 'competition_id', 'season'])
        .groupby(['competition_id', 'season'])
        .agg(
            season_start=('date', 'min'),
            season_end=('date', 'max'),
            partidos=('game_id', 'nunique')
        )
        .reset_index()
    )

    # Mostramos las últimas dos temporadas de algunas de las competiciones con más partidos
    top_competiciones = df_games['competition_id'].value_counts().head(8).index
    muestra_calendario = (
        df_season_calendar[df_season_calendar['competition_id'].isin(top_competiciones)]
        .sort_values(['competition_id', 'season_end'])
        .groupby('competition_id')
        .tail(2)
        .sort_values(['competition_id', 'season_end'])
    )
    print('Competiciones distintas:', df_games['competition_id'].nunique())
    print('Competición-temporadas:', len(df_season_calendar))
    print(muestra_calendario.to_string(index=False))
    print('Valoraciones con valor de mercado cero:', (df_valuations['market_value_in_eur'] == 0).sum())

## 5. Estadísticas por jugador, competición y temporada

Unimos las apariciones con los partidos para recuperar competición y temporada. Agregamos minutos, goles y asistencias por jugador/competición/temporada. Calculamos ratios por 90 más adelante, después de seleccionar las temporadas válidas.

### Por qué resumimos las apariciones por jugador, competición y temporada

`appearances.csv` tiene registros de partidos individuales. Para predecir un valor de mercado no necesitamos entregar al modelo una fila por cada partido: necesitamos un resumen del rendimiento reciente del jugador.

Por eso sumamos, para cada jugador y competición-temporada, los partidos, minutos, goles y asistencias. Mantenemos la competición separada en esta etapa porque cada competición tiene su propio calendario. Más adelante se seleccionarán las temporadas válidas para cada valoración y se sumarán las estadísticas que correspondan.


In [ ]:
# Esta sección solo se ejecuta si no hay caché válido; evita repetir cálculos pesados.
if not CACHE_CARGADO:
    # ==========================================================
    # 5. AGREGAR ESTADÍSTICAS POR JUGADOR-COMPETICIÓN-TEMPORADA
    # ==========================================================

    # Añadimos competición, temporada y fecha del partido a cada aparición
    df_appearances_season = df_appearances.merge(
        df_games[['game_id', 'competition_id', 'season', 'date']],
        on='game_id',
        how='left',
        validate='many_to_one',
        suffixes=('_appearance', '_game')
    )


    # Controles: apariciones sin temporada y discordancias de competición
    sin_partido = df_appearances_season['season'].isna().sum()
    comp_aparicion = df_appearances_season['competition_id_appearance'].astype('string')
    comp_partido = df_appearances_season['competition_id_game'].astype('string')
    discordancias_competicion = (
        comp_aparicion.notna() & comp_partido.notna() & comp_aparicion.ne(comp_partido)
    ).sum()
    print('Apariciones sin partido/temporada:', sin_partido)
    print('Discordancias de competición:', discordancias_competicion)

    # Usamos la competición del partido como referencia
    df_appearances_season['competition_id'] = df_appearances_season['competition_id_game']

    # Sumamos el rendimiento del jugador en cada competición-temporada
    df_stats_comp_season = (
        df_appearances_season.dropna(subset=['season', 'competition_id'])
        .groupby(['player_id', 'competition_id', 'season'])
        .agg(
            partidos=('appearance_id', 'nunique'),
            minutos=('minutes_played', 'sum'),
            goles=('goals', 'sum'),
            asistencias=('assists', 'sum')
        )
        .reset_index()
        .merge(
            df_season_calendar[['competition_id', 'season', 'season_end']],
            on=['competition_id', 'season'],
            how='left',
            validate='many_to_one'
        )
    )
    print('\nFilas jugador-comp.-temporada:', len(df_stats_comp_season))
    print('Jugadores incluidos:', df_stats_comp_season['player_id'].nunique())
    print('Registros sin fecha fin de temporada:', df_stats_comp_season['season_end'].isna().sum())
    print(df_stats_comp_season.head(5).round(2).to_string(index=False))

    # Liberamos la unión grande de apariciones para ahorrar memoria
    del df_appearances_season

## 6. Construir el dataset histórico `df2`

Creamos una fila por valoración positiva desde 2015. Para cada jugador y valoración buscamos, en cada competición, la última temporada terminada **antes** de la valoración. Usamos `merge_asof` para buscar la temporada anterior más cercana sin crear una unión gigante con todas las temporadas históricas.

Sumamos estadísticas de las competiciones seleccionadas y calculamos goles y asistencias por 90 minutos. La edad se calcula exactamente en la fecha de valoración.

### Decisión clave: construir una fotografía histórica sin mirar al futuro

Cada fila de `df2` representa **una valoración de un jugador en una fecha concreta**, no un jugador único para toda su carrera. Por eso un mismo `player_id` puede aparecer muchas veces, por ejemplo en 2021, 2022 y 2023, con un valor distinto en cada fecha.

Para cada valoración buscamos la temporada más reciente que ya había terminado antes de esa fecha. `direction='backward'` busca hacia atrás en el tiempo y `allow_exact_matches=False` excluye una temporada que terminase exactamente el mismo día de la valoración. Como solo tenemos fechas y no la hora exacta de los eventos, esta regla es prudente: evita asumir que las estadísticas de ese mismo día ya estaban disponibles.

Esta decisión es especialmente importante para evitar **fuga de información**: sería injusto predecir una valoración antigua utilizando goles o minutos de una temporada futura. La edad también se calcula en la fecha de valoración, no con la edad actual del jugador.

Los goles y las asistencias por 90 minutos se calculan después de sumar los minutos y las acciones de las competiciones seleccionadas. Así el ratio refleja el rendimiento global ponderado por los minutos jugados, en vez de dar el mismo peso a una competición en la que jugó mucho que a otra en la que jugó poco.

**Identificadores y posición:** `valuation_id` y `player_id` sirven para identificar y unir registros; no se usan como variables predictoras. `position` y `sub_position` sí se conservan como características. El notebook las toma de `players.csv` y no reconstruye un historial de cambios de posición; si esa tabla contiene solo la ficha actual, la posición podría no coincidir exactamente con la que tenía el jugador en una valoración antigua.


In [ ]:
# Esta sección solo se ejecuta si no hay caché válido; evita repetir cálculos pesados.
if not CACHE_CARGADO:
    # ==========================================================
    # 6. CONSTRUIR DF2 HISTÓRICO (UNA FILA POR VALORACIÓN)
    # ==========================================================

    # Base de valoraciones: positivas, con fecha y desde 2015
    df2_base = df_valuations.loc[
        (df_valuations['market_value_in_eur'] > 0)
        & (df_valuations['date'].notna())
        & (df_valuations['date'].dt.year >= 2015)
    ].copy()
    df2_base = df2_base.rename(columns={'date': 'valuation_date'}).reset_index(drop=True)
    df2_base['valuation_year'] = df2_base['valuation_date'].dt.year
    df2_base['valuation_id'] = np.arange(len(df2_base))

    # Añadimos fecha de nacimiento y posiciones desde players
    df2_base = df2_base.merge(
        df_players[['player_id', 'date_of_birth', 'position', 'sub_position']],
        on='player_id',
        how='left',
        validate='many_to_one'
    )
    df2_base['date_of_birth'] = pd.to_datetime(df2_base['date_of_birth'], errors='coerce')

    # Edad completa en la fecha de valoración (si no hay nacimiento, queda nula)
    val_date = df2_base['valuation_date']
    dob = df2_base['date_of_birth']
    birthday_not_reached = (
        (val_date.dt.month < dob.dt.month)
        | ((val_date.dt.month == dob.dt.month) & (val_date.dt.day < dob.dt.day))
    )
    df2_base['age'] = (
        val_date.dt.year - dob.dt.year - birthday_not_reached.astype('Int64')
    ).astype('Int64')

    # Pares jugador-competición en los que existen estadísticas
    # Expandimos cada valoración solo por las competiciones de ese jugador
    # (no por todas las temporadas históricas)
    df_player_competitions = (
        df_stats_comp_season.dropna(subset=['season_end'])[['player_id', 'competition_id']]
        .drop_duplicates()
    )
    df_val_comp = df2_base[['valuation_id', 'player_id', 'valuation_date']].merge(
        df_player_competitions,
        on='player_id',
        how='inner'
    )

    # Preparamos las estadísticas para buscar la temporada completada más reciente
    # Si dos temporadas tienen la misma fecha de fin para un jugador/competición, conservamos una
    df_stats_asof = (
        df_stats_comp_season.dropna(subset=['season_end'])
        .sort_values(['season_end', 'player_id', 'competition_id', 'season'])
        .drop_duplicates(['player_id', 'competition_id', 'season_end'], keep='last')
    )
    df_val_comp = df_val_comp.sort_values(['valuation_date', 'player_id', 'competition_id']).reset_index(drop=True)
    df_stats_asof = df_stats_asof.sort_values(['season_end', 'player_id', 'competition_id']).reset_index(drop=True)

    # Seleccionamos la temporada más reciente que terminó antes de cada valoración
    # allow_exact_matches=False excluye temporadas que terminan el mismo día de la valoración
    df_snapshot_comp = pd.DataFrame()
    if not df_val_comp.empty and not df_stats_asof.empty:
        df_snapshot_comp = pd.merge_asof(
            df_val_comp,
            df_stats_asof,
            left_on='valuation_date',
            right_on='season_end',
            by=['player_id', 'competition_id'],
            direction='backward',
            allow_exact_matches=False
        )
        df_snapshot_comp = df_snapshot_comp.dropna(subset=['season_end'])
        df_stats_snapshot = (
            df_snapshot_comp.groupby('valuation_id')
            .agg(
                competiciones_con_stats=('competition_id', 'nunique'),
                latest_stats_end=('season_end', 'max'),
                partidos=('partidos', 'sum'),
                minutos=('minutos', 'sum'),
                goles=('goles', 'sum'),
                asistencias=('asistencias', 'sum')
            )
            .reset_index()
        )
    else:
        # Tabla vacía con tipos de datos compatibles para el merge
        df_stats_snapshot = pd.DataFrame({
            'valuation_id': pd.Series(dtype='int64'),
            'competiciones_con_stats': pd.Series(dtype='float64'),
            'latest_stats_end': pd.Series(dtype='datetime64[ns]'),
            'partidos': pd.Series(dtype='float64'),
            'minutos': pd.Series(dtype='float64'),
            'goles': pd.Series(dtype='float64'),
            'asistencias': pd.Series(dtype='float64')
        })

    # Unimos las estadísticas a cada valoración; las que no tienen estadísticas quedan nulas
    df2 = df2_base.merge(df_stats_snapshot, on='valuation_id', how='left', validate='one_to_one')

    # Ratios por 90 minutos; no se calculan si los minutos son nulos o cero
    minutos_validos = df2['minutos'].where(df2['minutos'] > 0)
    df2['goles_por90'] = df2['goles'] / minutos_validos * 90
    df2['asistencias_por90'] = df2['asistencias'] / minutos_validos * 90

    # Conservamos variables útiles para el análisis y la futura unión con GMM
    columnas_df2 = [
        'valuation_id', 'player_id', 'valuation_date', 'valuation_year',
        'age', 'position', 'sub_position', 'competiciones_con_stats', 'latest_stats_end',
        'partidos', 'minutos', 'goles', 'asistencias', 'goles_por90', 'asistencias_por90',
        'market_value_in_eur'
    ]
    df2 = df2[columnas_df2].copy()

    # Comprobamos que las estadísticas usadas sean anteriores a la valoración
    mask_stats = df2['latest_stats_end'].notna()
    assert (df2.loc[mask_stats, 'latest_stats_end'] < df2.loc[mask_stats, 'valuation_date']).all()
    assert df2.duplicated(['player_id', 'valuation_date']).sum() == 0

    print('DF2 creado')
    print('Valoraciones:', len(df2))
    print('Jugadores:', df2['player_id'].nunique())
    print('Duplicados jugador-fecha:', df2.duplicated(['player_id', 'valuation_date']).sum())
    print('Con estadísticas:', df2['minutos'].notna().sum())
    print('Sin estadísticas:', df2['minutos'].isna().sum())
    print('Edades nulas:', df2['age'].isna().sum())
    print(df2.head(5).round(3).to_string(index=False))

    # Liberamos tablas intermedias grandes
    del df_val_comp, df_stats_asof, df_snapshot_comp, df_player_competitions, df_stats_snapshot, df2_base

## 7. Cobertura y filtros del dataset de regresión

Medimos qué porcentaje de valoraciones tiene estadísticas. Después aplicamos filtros secuenciales y guardamos una auditoría con las filas eliminadas en cada paso.

Filtros: valor de mercado positivo, estadísticas con minutos mayores que cero, edad entre 15 y 45, subposición conocida y ratios por 90 válidos. **No exigimos 450 minutos**, porque ese umbral pertenece al clustering, no a esta regresión.

### Por qué aplicamos estos filtros y por qué no exigimos 450 minutos

Los filtros definen qué observaciones tienen información suficiente para esta primera regresión:

- **Valor de mercado positivo:** necesitamos un objetivo real y utilizable; se descartan valores nulos, cero o negativos.
- **Minutos mayores que cero:** sin minutos no se pueden calcular goles y asistencias por 90.
- **Edad entre 15 y 45 años:** acotamos la población a edades razonables para este conjunto de futbolistas y evitamos edades extremas que podrían ser errores. Es una decisión de alcance, no una regla universal.
- **Subposición conocida:** necesitamos una categoría de posición para que el modelo pueda aprender diferencias entre roles.
- **Ratios por 90 válidos:** quitamos valores nulos o infinitos que no se podrían usar correctamente en el entrenamiento.

**No aplicamos el mínimo de 450 minutos del clustering.** Aquí preferimos conservar valoraciones que tengan estadísticas válidas, aunque el jugador haya jugado poco. La contrapartida es que un ratio por 90 calculado con pocos minutos puede ser inestable: por ejemplo, un gol en muy poco tiempo puede producir un valor por 90 exagerado. Es una limitación que se podría estudiar en un experimento de sensibilidad posterior.


In [ ]:
# Esta sección solo se ejecuta si no hay caché válido; evita repetir cálculos pesados.
if not CACHE_CARGADO:
    # ==========================================================
    # 7. COBERTURA Y FILTROS
    # ==========================================================

    # Cobertura de estadísticas por año
    df2['tiene_stats'] = df2['minutos'].notna()
    df_cobertura_anual = (
        df2.groupby('valuation_year')
        .agg(valoraciones=('player_id', 'size'), con_stats=('tiene_stats', 'sum'))
        .reset_index()
    )
    df_cobertura_anual['porcentaje_con_stats'] = (
        df_cobertura_anual['con_stats'] / df_cobertura_anual['valoraciones'] * 100
    )
    print('Cobertura total de estadísticas:', round(df2['tiene_stats'].mean() * 100, 2), '%')
    print(df_cobertura_anual.round(2).to_string(index=False))

    # Filtros en orden, con auditoría de filas antes y después
    df2_regression = df2.copy()
    auditoria_filtros = []

    def aplicar_filtro(df, condicion, nombre):
        antes = len(df)
        df_filtrado = df.loc[condicion].copy()
        auditoria_filtros.append({
            'Filtro': nombre,
            'Antes': antes,
            'Después': len(df_filtrado),
            'Eliminadas': antes - len(df_filtrado)
        })
        return df_filtrado

    # 1. Valor de mercado positivo
    df2_regression = aplicar_filtro(
        df2_regression,
        df2_regression['market_value_in_eur'].notna() & (df2_regression['market_value_in_eur'] > 0),
        'Valor de mercado positivo'
    )
    # 2. Estadísticas disponibles y minutos positivos
    df2_regression = aplicar_filtro(
        df2_regression,
        df2_regression['minutos'].notna() & (df2_regression['minutos'] > 0),
        'Estadísticas y minutos > 0'
    )
    # 3. Edad razonable para la población de futbolistas
    df2_regression = aplicar_filtro(
        df2_regression,
        df2_regression['age'].between(15, 45),
        'Edad entre 15 y 45 años'
    )
    # 4. Subposición conocida
    df2_regression = aplicar_filtro(
        df2_regression,
        df2_regression['sub_position'].notna()
        & df2_regression['sub_position'].astype('string').str.strip().ne(''),
        'Subposición conocida'
    )
    # 5. Ratios por 90 válidos (sin nulos ni infinitos)
    ratios_validos = (
        df2_regression['goles_por90'].notna()
        & df2_regression['asistencias_por90'].notna()
        & np.isfinite(df2_regression[['goles_por90', 'asistencias_por90']]).all(axis=1)
    )
    df2_regression = aplicar_filtro(df2_regression, ratios_validos, 'Ratios por 90 válidos')

    # Mostrar auditoría y comprobar que jugador-fecha sea única
    df_auditoria_filtros = pd.DataFrame(auditoria_filtros)
    print('\nAUDITORÍA DE FILTROS')
    print(df_auditoria_filtros.to_string(index=False))
    print('\nDataset final para regresión')
    print('Filas:', len(df2_regression))
    print('Jugadores:', df2_regression['player_id'].nunique())
    print('Período:', df2_regression['valuation_date'].min().date(), 'a', df2_regression['valuation_date'].max().date())
    print('Duplicados jugador-fecha:', df2_regression.duplicated(['player_id', 'valuation_date']).sum())
    assert df2_regression.duplicated(['player_id', 'valuation_date']).sum() == 0

    # Guardamos el dataset ya preparado y los resúmenes necesarios para futuras ejecuciones.
    df2_regression.to_csv(CACHE_DF2, index=False)
    df_auditoria_filtros.to_csv(CACHE_AUDITORIA, index=False)
    df_cobertura_anual.to_csv(CACHE_COBERTURA, index=False)
    with open(CACHE_METADATA, 'w', encoding='utf-8') as f:
        json.dump({'firma_fuentes': FIRMA_FUENTES}, f, ensure_ascii=False, indent=2)
    print('\nCaché creado en Drive. En las siguientes ejecuciones se saltará esta preparación pesada.')


## 8. Distribución del objetivo después de los filtros

Revisamos la distribución del valor de mercado en el dataset que realmente entra al modelo. Sigue siendo muy desigual, por eso el histograma se muestra en escala logarítmica.

In [ ]:
# ==========================================================
# 8. DISTRIBUCIÓN FINAL DEL VALOR DE MERCADO
# ==========================================================

valores_finales_millones = df2_regression['market_value_in_eur'] / 1_000_000
print(valores_finales_millones.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(3))

if len(valores_finales_millones) > 0:
    bins_finales = np.logspace(
        np.log10(valores_finales_millones.min()),
        np.log10(valores_finales_millones.max()),
        40
    )
    plt.figure(figsize=(9, 4))
    plt.hist(valores_finales_millones, bins=bins_finales)
    plt.xscale('log')
    plt.title('Valor de mercado en DF2 de regresión')
    plt.xlabel('Millones de euros (escala logarítmica)')
    plt.ylabel('Valoraciones')
    plt.grid(axis='y')
    plt.tight_layout()
    plt.show()

### Qué nos dice la distribución final

En la salida de esta ejecución hay **291.962 valoraciones**. La media es **3,659 M€**, pero la mediana es solo **0,750 M€**. Que la media sea bastante mayor que la mediana indica que unos pocos jugadores con valores muy altos empujan la media hacia arriba.

Además, el percentil 90 es **9 M€**, el percentil 99 es **45 M€** y el máximo llega a **200 M€**. Es una distribución con una cola larga: la mayoría de las valoraciones son mucho menores que las más caras. Por eso usamos escala logarítmica en el gráfico, para poder ver mejor toda la distribución. El entrenamiento y las métricas, sin embargo, continúan en euros originales.


## 9. Separación temporal con TimeSeriesSplit y definición de variables

Reservamos primero el **Test final (2024 en adelante)**. Los datos hasta 2023 se usan para comparar modelos mediante `TimeSeriesSplit`.

Aplicamos `TimeSeriesSplit` a la lista de **años únicos**, no directamente a las filas. Así todas las valoraciones de un mismo año permanecen juntas y no repartimos un mismo año entre Train y Validation.

Usamos tres pliegues (`n_splits=3`) y validamos un año completo en cada pliegue (`test_size=1`). Con datos de 2015 a 2023, por ejemplo, se validan 2021, 2022 y 2023, cada uno después de entrenar con los años anteriores.

Después de elegir el modelo por el MAE medio de los tres pliegues, lo entrenamos con todos los datos hasta 2023 y lo evaluamos una sola vez en Test (2024 en adelante).

### Por qué elegimos estas variables y esta separación temporal

El objetivo (`target_col`) es el valor de mercado en euros. Las variables de entrada se han elegido porque aportan información futbolística sencilla y disponible antes de la valoración:

| Variable | Qué aporta al modelo |
|---|---|
| `age` | La etapa de carrera del jugador: edad y potencial de desarrollo pueden influir en su valor. |
| `minutos` | Cuánto ha participado; ayuda a distinguir a un jugador habitual de uno con poca participación. |
| `goles_por90` | Producción goleadora ajustada al tiempo jugado, para comparar jugadores con minutos distintos. |
| `asistencias_por90` | Aportación en asistencias, también normalizada por minutos. |
| `position`, `sub_position` | El papel del jugador; no se espera el mismo rendimiento estadístico ni el mismo valor para todos los puestos. |
| `valuation_year` (opcional) | Permite comprobar si conocer el año mejora la predicción ante cambios generales del mercado. No es una corrección de inflación. |

No usamos `player_id` para que el modelo no dependa de memorizar la identidad de cada futbolista; queremos que aprenda patrones de edad, participación, producción y posición. Tampoco usamos `valuation_date` como entrada: se utiliza para ordenar y separar los periodos, no como característica del modelo.

Se reserva **2015–2023** para desarrollar y elegir el modelo, y **2024 en adelante** para el test final. Esta separación responde a la pregunta real del proyecto: si entrenamos con valoraciones antiguas, ¿cómo funciona el modelo en valoraciones posteriores? Una separación aleatoria mezclaría fechas antiguas y futuras en ambos conjuntos y daría una prueba menos parecida a ese uso futuro.


In [ ]:
# ==========================================================
# 9. CREAR LOS CONJUNTOS TEMPORALES Y DEFINIR FEATURES
# ==========================================================

from sklearn.model_selection import TimeSeriesSplit

# Variables que usará la regresión.
features_numericas = ['age', 'minutos', 'goles_por90', 'asistencias_por90']
features_categoricas = ['position', 'sub_position']
features_sin_anio = features_numericas + features_categoricas
features_con_anio = features_sin_anio + ['valuation_year']
target_col = 'market_value_in_eur'

# Reservamos desde el principio el Test final: nunca se usa para elegir el modelo.
df_train_candidatos = (
    df2_regression.loc[df2_regression['valuation_year'] <= 2023]
    .sort_values('valuation_date')
    .copy()
)
df_train_final = df_train_candidatos.copy()
df_test = (
    df2_regression.loc[df2_regression['valuation_year'] >= 2024]
    .sort_values('valuation_date')
    .copy()
)

# Comprobamos que hay datos en ambos periodos y que Train es anterior a Test.
assert not df_train_candidatos.empty, 'No hay datos para entrenar hasta 2023.'
assert not df_test.empty, 'No hay datos de Test desde 2024.'
assert df_train_candidatos['valuation_year'].max() <= 2023
assert df_test['valuation_year'].min() >= 2024
assert df_train_final['valuation_date'].max() < df_test['valuation_date'].min()

# TimeSeriesSplit trabaja aquí con años únicos, para que un año no se divida entre conjuntos.
anios_cv = np.sort(
    df_train_candidatos['valuation_year'].dropna().astype(int).unique()
)
N_SPLITS_CV = 3
TEST_SIZE_CV = 1  # Cada pliegue valida un año completo.

if len(anios_cv) == 0:
    raise ValueError('No hay años de valoración válidos hasta 2023.')

if len(anios_cv) <= N_SPLITS_CV:
    raise ValueError(
        f'Se necesitan al menos {N_SPLITS_CV + 1} años hasta 2023 para crear '
        f'{N_SPLITS_CV} pliegues. Años encontrados: {anios_cv.tolist()}'
    )

# Creamos los pliegues: el entrenamiento crece y la validación avanza en el tiempo.
divisor_temporal = TimeSeriesSplit(
    n_splits=N_SPLITS_CV,
    test_size=TEST_SIZE_CV
)

pliegues_cv = []
for numero_pliegue, (idx_train, idx_val) in enumerate(divisor_temporal.split(anios_cv), start=1):
    anios_train = anios_cv[idx_train].astype(int).tolist()
    anios_val = anios_cv[idx_val].astype(int).tolist()

    # Guardamos los años; en la siguiente sección seleccionaremos las filas correspondientes.
    pliegues_cv.append({
        'fold': numero_pliegue,
        'train_years': anios_train,
        'validation_years': anios_val
    })

# Mostramos exactamente qué años se usan en cada evaluación.
print('ESTRATEGIA: TimeSeriesSplit por año')
print(f'Número de pliegues: {N_SPLITS_CV} | Años de validación por pliegue: {TEST_SIZE_CV}')
for pliegue in pliegues_cv:
    filas_train = df_train_candidatos['valuation_year'].isin(pliegue['train_years']).sum()
    filas_val = df_train_candidatos['valuation_year'].isin(pliegue['validation_years']).sum()
    print(
        f"Pliegue {pliegue['fold']}: "
        f"Train {pliegue['train_years'][0]}–{pliegue['train_years'][-1]} "
        f"({filas_train:,} filas) | "
        f"Validation {pliegue['validation_years']} ({filas_val:,} filas)"
    )

# Avisamos si falta algún año en el periodo histórico; los pliegues usan los años disponibles.
if not np.array_equal(anios_cv, np.arange(anios_cv.min(), anios_cv.max() + 1)):
    print('Aviso: hay años sin valoraciones. TimeSeriesSplit usa los años disponibles en orden.')

print(
    '\nTrain final:',
    df_train_final['valuation_date'].min().date(), 'a',
    df_train_final['valuation_date'].max().date()
)
print(
    'Test final:',
    df_test['valuation_date'].min().date(), 'a',
    df_test['valuation_date'].max().date()
)
print('\nFeatures sin año:', features_sin_anio)
print('Features con año:', features_con_anio)
print('Objetivo:', target_col)

### Cómo leer los periodos que ha creado el notebook

La salida confirma tres validaciones temporales:

| Pliegue | Train disponible | Año de Validation |
|---|---:|---:|
| 1 | 2015–2020 (138.422 filas) | 2021 (33.462 filas) |
| 2 | 2015–2021 (171.884 filas) | 2022 (32.385 filas) |
| 3 | 2015–2022 (204.269 filas) | 2023 (30.834 filas) |

El entrenamiento avanza año a año y la validación siempre está en el futuro respecto a Train. Todas las valoraciones de un mismo año se mantienen juntas. El test final va del **3 de enero de 2024 al 11 de junio de 2026**, según las fechas que hay en este notebook; por tanto, 2026 es un periodo parcial, no un año completo.

Más adelante el código limita las filas usadas para entrenar cada pliegue a 60.000. Los números de Train de esta tabla son las filas disponibles antes de aplicar ese límite.


## 10. Preprocesamiento y modelos

El preprocesamiento se ajusta dentro de cada pipeline, solo con Train, para evitar fuga de información:

- Variables numéricas: imputación por mediana; Ridge también escala estas columnas.
- Variables categóricas: imputación por valor más frecuente y One-Hot Encoding.
- Modelos: baseline de media, Ridge y Random Forest.
- Comparamos modelos sin año y con `valuation_year`.

Es una comparación inicial, no una búsqueda exhaustiva de hiperparámetros.

**Nota de rendimiento:** Random Forest se configura en versión ligera (20 árboles, profundidad 10 y muestreo del 50%) para que la comparación se ejecute más rápido en Colab. Es una configuración inicial, no una optimización exhaustiva.


### Por qué usamos este preprocesamiento y estos modelos

Antes de entrenar, hay que preparar los datos de forma adecuada:

- En las variables numéricas, los valores que falten se rellenan con la **mediana** de Train. La mediana es menos sensible a valores extremos que la media.
- En las variables categóricas, los valores ausentes se rellenan con la categoría más frecuente y después se aplica **One-Hot Encoding**. Esto convierte, por ejemplo, las posiciones en columnas indicadoras, sin dar a “delantero” un número que parezca mayor o menor que “defensa”.
- `handle_unknown='ignore'` permite que el modelo no falle si en validación o en una futura predicción aparece una categoría que no vio durante el entrenamiento.
- **Ridge** escala las variables numéricas porque es un modelo lineal y la escala de las variables puede afectar a su ajuste. **Random Forest** no necesita ese escalado: toma decisiones mediante cortes o umbrales en cada variable.
- Todo está dentro de un `Pipeline`: el relleno de nulos, la codificación y el escalado se aprenden solo con Train en cada pliegue. Así Validation y Test no participan en el ajuste del preprocesamiento y evitamos otra forma de fuga de información.

Comparamos tres referencias con funciones distintas: **Dummy media** predice siempre la media de los valores de Train y sirve como mínimo de referencia; **Ridge** comprueba qué consigue un modelo lineal sencillo; y **Random Forest** combina árboles y puede aprender relaciones no lineales e interacciones entre edad, minutos, ratios y posición. Ridge es un comparador, no el modelo que se ha seleccionado en esta ejecución.


In [ ]:
# ==========================================================
# 10. PREPROCESAMIENTO Y DEFINICIÓN DE MODELOS
# ==========================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow No se instala ni inicia servidor desde Colab.
MLFLOW_AVAILABLE = False
try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    mlflow.set_tracking_uri(MLFLOW_DIR.as_uri())
    mlflow.set_experiment('OFFBALL_Regresion_Valor_Mercado')
    print('MLflow disponible: tracking local en', mlflow.get_tracking_uri())
except ModuleNotFoundError:
    print('MLflow no está disponible. Se entrenará y guardará localmente, sin tracking de runs.')
except Exception as error:
    MLFLOW_AVAILABLE = False
    print('No se pudo activar MLflow:', type(error).__name__, '| Se continúa sin tracking.')

# Límites de entrenamiento para que la ejecución sea manejable en Colab.
CV_TRAIN_MAX_ROWS = 60000  # máximo de filas de Train por pliegue, común a todos los modelos
RF_FINAL_MAX_ROWS = 100000      # tope de entrenamiento del RF final
RF_ESTIMATORS = 20
RF_MAX_DEPTH = 10
RF_MIN_LEAF = 15
RF_MAX_FEATURES = 0.5
RF_MAX_SAMPLES = 0.5


def crear_preprocesador(features, escalar_numericas=False):
    columnas_num = [c for c in features if c in features_numericas or c == 'valuation_year']
    columnas_cat = [c for c in features if c in features_categoricas]
    transformadores = []
    if columnas_num:
        pasos_num = [('imputar', SimpleImputer(strategy='median'))]
        if escalar_numericas:
            pasos_num.append(('escalar', StandardScaler()))
        transformadores.append(('numericas', Pipeline(steps=pasos_num), columnas_num))
    if columnas_cat:
        transformadores.append((
            'categoricas',
            Pipeline(steps=[
                ('imputar', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]),
            columnas_cat
        ))
    return ColumnTransformer(transformers=transformadores, remainder='drop')


def crear_modelo(tipo, features):
    if tipo == 'dummy':
        return DummyRegressor(strategy='mean')
    if tipo == 'ridge':
        return Pipeline(steps=[
            ('preprocesado', crear_preprocesador(features, escalar_numericas=True)),
            ('modelo', Ridge(alpha=1.0))
        ])
    if tipo == 'rf':
        # RF pequeño y acotado: baseline rápida, no búsqueda exhaustiva.
        return Pipeline(steps=[
            ('preprocesado', crear_preprocesador(features, escalar_numericas=False)),
            ('modelo', RandomForestRegressor(
                n_estimators=RF_ESTIMATORS,
                max_depth=RF_MAX_DEPTH,
                min_samples_leaf=RF_MIN_LEAF,
                max_features=RF_MAX_FEATURES,
                max_samples=RF_MAX_SAMPLES,
                random_state=42,
                n_jobs=-1
            ))
        ])
    raise ValueError(f'Tipo de modelo desconocido: {tipo}')



configuraciones = [
    ('Dummy media', 'dummy', []),
    ('Ridge sin año', 'ridge', features_sin_anio),
    ('Ridge con año', 'ridge', features_con_anio),
    ('Random Forest sin año', 'rf', features_sin_anio),
    ('Random Forest con año', 'rf', features_con_anio)
]


def calcular_metricas(y_real, y_pred):
    return {
        'MAE_MEUR': mean_absolute_error(y_real, y_pred) / 1_000_000,
        'RMSE_MEUR': np.sqrt(mean_squared_error(y_real, y_pred)) / 1_000_000,
        'R2': r2_score(y_real, y_pred)
    }

print('Modelos preparados.')
print(f'Cada pliegue usa hasta {CV_TRAIN_MAX_ROWS:,} filas de Train; Validation usa todas las filas del año.')
print(f'Random Forest: {RF_ESTIMATORS} árboles, profundidad {RF_MAX_DEPTH}, hoja mínima {RF_MIN_LEAF}, max_samples={RF_MAX_SAMPLES}.')
print(f'RF final: máximo {RF_FINAL_MAX_ROWS:,} filas; paralelizado con todos los cores disponibles.')
print('MLflow activo:', MLFLOW_AVAILABLE)


### Por qué Random Forest tiene estos parámetros

La configuración se mantiene deliberadamente ligera para poder ejecutar la comparación en Colab sin convertir esta fase en una búsqueda larga de parámetros:

- **20 árboles (`n_estimators=20`):** combina varios árboles, pero limita el coste de cálculo. Más árboles podrían mejorar la estabilidad, a cambio de más tiempo.
- **Profundidad máxima 10 (`max_depth=10`):** limita lo complejas que pueden ser las reglas de cada árbol y ayuda a controlar el sobreajuste.
- **Mínimo de 15 observaciones por hoja (`min_samples_leaf=15`):** evita que una predicción dependa de grupos minúsculos de ejemplos.
- **50 % de variables (`max_features=0.5`):** cada división considera una parte de las variables; esto aporta diversidad entre los árboles.
- **50 % de muestras por árbol (`max_samples=0.5`):** reduce el coste de cada árbol y hace que no todos aprendan exactamente de la misma muestra.
- **`random_state=42`:** fija la aleatoriedad para poder repetir la comparación en condiciones equivalentes.
- **`n_jobs=-1`:** utiliza los núcleos disponibles para paralelizar el cálculo.

Estos valores son un punto de partida razonable para esta ejecución, no unos parámetros demostrados como óptimos. Por eso describimos el resultado como una **línea base** y no como el máximo rendimiento posible de Random Forest.


## 11. Entrenar y comparar con TimeSeriesSplit

Para cada uno de los tres pliegues:

1. Entrenamos con los años anteriores.
2. Validamos con el siguiente año completo.
3. Repetimos el proceso con las cinco configuraciones.

En cada pliegue usamos la misma muestra reproducible de hasta 60.000 filas de entrenamiento para comparar los modelos en condiciones equivalentes. La validación utiliza todas las filas del año correspondiente.

Elegimos la configuración con el **MAE medio más bajo** entre los pliegues; el RMSE medio sirve como desempate. Si MLflow está disponible, registramos cada ejecución. El Test final (2024 en adelante) no participa en esta selección.

### Por qué validamos de esta manera y qué significan las métricas

`TimeSeriesSplit` simula varias veces el uso futuro del modelo: se entrena con años anteriores y se comprueba si predice el año siguiente. Se repite en 2021, 2022 y 2023 para no depender de un único año de validación.

En cada pliegue se toma una muestra reproducible de hasta **60.000 filas de Train**, y se utiliza la misma muestra para las cinco configuraciones. Esto reduce el tiempo de Colab y permite compararlas en condiciones equivalentes. Validation utiliza todas las filas del año correspondiente, no una muestra.

Las métricas responden a preguntas distintas:

- **MAE:** cuánto se equivoca el modelo de media en valor absoluto. Es la métrica principal porque se entiende directamente en millones de euros y no permite que los errores positivos y negativos se cancelen.
- **RMSE:** también mide el error, pero penaliza más los fallos muy grandes. Es útil porque el mercado tiene jugadores de valores muy distintos.
- **R²:** compara el modelo con una predicción constante basada en la media y resume qué parte de la variabilidad consigue explicar. Puede ser negativo si el modelo funciona peor que esa referencia. **No es un porcentaje de aciertos ni una medida de exactitud.**

Elegimos por el MAE medio de los tres pliegues y usamos el RMSE medio como desempate. Así priorizamos reducir el error absoluto típico, sin dejar de vigilar los errores grandes.


In [ ]:
# ==========================================================
# 11. ENTRENAR Y COMPARAR LOS MODELOS CON TIMESERIESSPLIT
# ==========================================================

from contextlib import nullcontext

# Guardaremos una fila por modelo y por pliegue.
resultados_folds = []
resumen_folds_cv = []

for pliegue in pliegues_cv:
    numero_pliegue = pliegue['fold']
    anios_train = pliegue['train_years']
    anios_val = pliegue['validation_years']

    # Separamos las filas usando los años definidos por TimeSeriesSplit.
    df_train_fold = df_train_candidatos[
        df_train_candidatos['valuation_year'].isin(anios_train)
    ].copy()
    df_val_fold = df_train_candidatos[
        df_train_candidatos['valuation_year'].isin(anios_val)
    ].copy()

    # Usamos una muestra de entrenamiento común para todos los modelos de este pliegue.
    df_train_fold_modelo = df_train_fold.sample(
        n=min(CV_TRAIN_MAX_ROWS, len(df_train_fold)),
        random_state=42
    ).copy()

    resumen_folds_cv.append({
        'fold': int(numero_pliegue),
        'train_years': [int(anio) for anio in anios_train],
        'validation_years': [int(anio) for anio in anios_val],
        'train_rows_total': int(len(df_train_fold)),
        'train_rows_sampled': int(len(df_train_fold_modelo)),
        'validation_rows': int(len(df_val_fold))
    })

    print(
        f"\nPLIEGUE {numero_pliegue} | "
        f"Train {anios_train[0]}–{anios_train[-1]} "
        f"({len(df_train_fold_modelo):,} filas usadas) | "
        f"Validation {anios_val} ({len(df_val_fold):,} filas)"
    )

    # Comparamos las mismas cinco configuraciones en cada pliegue.
    for nombre, tipo, features in configuraciones:
        print(f'  Entrenando: {nombre}...')
        modelo = crear_modelo(tipo, features)

        if tipo == 'dummy':
            # Dummy no utiliza variables: predice la media del valor de Train.
            X_train = np.zeros((len(df_train_fold_modelo), 1))
            X_val = np.zeros((len(df_val_fold), 1))
        else:
            # Preparamos las columnas del modelo; el Pipeline ajusta el preprocesamiento solo con Train.
            X_train = df_train_fold_modelo[features].copy()
            X_val = df_val_fold[features].copy()
            columnas_num_modelo = [
                c for c in features
                if c in features_numericas or c == 'valuation_year'
            ]
            for conjunto in [X_train, X_val]:
                conjunto[columnas_num_modelo] = conjunto[columnas_num_modelo].apply(
                    pd.to_numeric, errors='coerce'
                )

        # Cada pliegue se registra como una ejecución independiente en MLflow, si está disponible.
        nombre_run = f'CV{numero_pliegue} - {nombre}'
        contexto_run = mlflow.start_run(run_name=nombre_run) if MLFLOW_AVAILABLE else nullcontext()

        with contexto_run:
            modelo.fit(X_train, df_train_fold_modelo[target_col])
            pred_val = modelo.predict(X_val)
            metricas = calcular_metricas(df_val_fold[target_col], pred_val)
            run_id = mlflow.active_run().info.run_id if MLFLOW_AVAILABLE else None

            if MLFLOW_AVAILABLE:
                mlflow.log_params({
                    'estrategia_validacion': 'TimeSeriesSplit',
                    'pliegue': int(numero_pliegue),
                    'anios_train': ','.join(map(str, anios_train)),
                    'anios_validation': ','.join(map(str, anios_val)),
                    'tipo_modelo': tipo,
                    'features': ', '.join(features) if features else 'baseline media',
                    'target': target_col,
                    'train_rows_total': len(df_train_fold),
                    'train_rows_sampled': len(df_train_fold_modelo),
                    'validation_rows': len(df_val_fold),
                    'alpha': 1.0 if tipo == 'ridge' else 'n/a',
                    'n_estimators': RF_ESTIMATORS if tipo == 'rf' else 'n/a',
                    'max_depth': RF_MAX_DEPTH if tipo == 'rf' else 'n/a',
                    'max_samples': RF_MAX_SAMPLES if tipo == 'rf' else 'n/a'
                })
                mlflow.log_metrics({
                    'cv_MAE_MEUR': metricas['MAE_MEUR'],
                    'cv_RMSE_MEUR': metricas['RMSE_MEUR'],
                    'cv_R2': metricas['R2']
                })

        # Guardamos el resultado individual para poder analizar cada periodo.
        resultados_folds.append({
            'Fold': numero_pliegue,
            'Train_years': ','.join(map(str, anios_train)),
            'Validation_years': ','.join(map(str, anios_val)),
            'Modelo': nombre,
            'Tipo': tipo,
            'Features': ', '.join(features) if features else 'baseline media',
            'MAE_MEUR': metricas['MAE_MEUR'],
            'RMSE_MEUR': metricas['RMSE_MEUR'],
            'R2': metricas['R2'],
            'MLflow_run_id': run_id
        })
        print(
            f"    MAE={metricas['MAE_MEUR']:.3f} M€ | "
            f"RMSE={metricas['RMSE_MEUR']:.3f} M€ | R²={metricas['R2']:.3f}"
        )

# Resultados detallados: una fila por modelo y por pliegue.
df_resultados_folds = pd.DataFrame(resultados_folds)

# Resumen: media de las métricas de los tres periodos; la desviación muestra su variación.
df_resultados_validacion = (
    df_resultados_folds
    .groupby(['Modelo', 'Tipo', 'Features'], as_index=False)
    .agg(
        MAE_MEUR=('MAE_MEUR', 'mean'),
        MAE_MEUR_std=('MAE_MEUR', 'std'),
        RMSE_MEUR=('RMSE_MEUR', 'mean'),
        RMSE_MEUR_std=('RMSE_MEUR', 'std'),
        R2=('R2', 'mean'),
        N_folds=('Fold', 'nunique')
    )
    .sort_values(['MAE_MEUR', 'RMSE_MEUR'])
    .reset_index(drop=True)
)

print('\nRESULTADOS DETALLADOS POR PLIEGUE')
print(
    df_resultados_folds[
        ['Fold', 'Modelo', 'Validation_years', 'MAE_MEUR', 'RMSE_MEUR', 'R2']
    ].round(3).to_string(index=False)
)

print('\nRESUMEN MEDIO DE VALIDACIÓN (MENOR MAE MEDIO = MEJOR)')
print(
    df_resultados_validacion[
        ['Modelo', 'MAE_MEUR', 'MAE_MEUR_std', 'RMSE_MEUR', 'R2', 'N_folds']
    ].round(3).to_string(index=False)
)

### Interpretación de la comparación de modelos

Estos son los promedios de los tres años de Validation que aparecen en la ejecución:

| Configuración | MAE medio | RMSE medio | R² medio |
|---|---:|---:|---:|
| **Random Forest sin año** | **2,670 M€** | 6,726 M€ | 0,409 |
| Random Forest con año | 2,705 M€ | **6,642 M€** | **0,423** |
| Ridge sin año | 3,703 M€ | 7,759 M€ | 0,212 |
| Ridge con año | 3,855 M€ | 7,766 M€ | 0,211 |
| Dummy media | 4,152 M€ | 8,747 M€ | -0,001 |

**Gana Random Forest sin año** porque su MAE medio es el más bajo: 2,670 M€. El Random Forest con año obtiene un RMSE y un R² ligeramente mejores, pero su MAE es 0,035 M€ (unos **35.000 €**) mayor. Como habíamos definido el MAE como criterio principal, se selecciona la versión sin año; no se elige mirando el Test final.

Random Forest mejora claramente frente a Ridge y Dummy en MAE. Esto indica que, con estas variables y estos datos, el modelo basado en árboles está captando patrones que el modelo lineal sencillo no recoge tan bien. Dummy confirma que no basta con predecir siempre la media.

El R² de 0,409 de la configuración ganadora en Validation no significa “40,9 % de acierto”. Indica el ajuste relativo a la variabilidad del objetivo en esos años. La diferencia entre los dos Random Forest es pequeña, así que conviene tratar esta elección como la mejor entre las configuraciones probadas, no como una conclusión universal.


## 12. Elegir configuración, reentrenar y evaluar el Test final

Elegimos la configuración con el **MAE medio más bajo** en los pliegues de `TimeSeriesSplit` (el RMSE medio desempata).

Después entrenamos de nuevo esa configuración con todos los datos hasta 2023. Para limitar el tiempo, Random Forest utiliza una muestra reproducible de hasta 100.000 filas; Ridge y Dummy utilizan todas las filas.

Por último, evaluamos el modelo una sola vez con las valoraciones de 2024 en adelante. **El Test final no se utiliza para elegir ni ajustar el modelo.**

### Por qué volvemos a entrenar antes del Test final

Una vez elegida la configuración con Validation, la volvemos a ajustar con los datos disponibles hasta el 30 de diciembre de 2023. Es razonable aprovechar todos los años de desarrollo (2015–2023) antes de hacer la prueba final, porque la decisión sobre el modelo ya se ha tomado.

Hay **235.103 filas disponibles** para ese entrenamiento final, pero Random Forest utiliza como máximo **100.000** con `random_state=42`. Es un compromiso para limitar memoria y tiempo en Colab; equivale a entrenar con una muestra reproducible de las filas disponibles, no con todas. Es importante documentarlo porque el modelo final no ha visto las 235.103 filas.

El Test contiene las valoraciones desde 2024. No se usa para seleccionar el modelo ni ajustar sus parámetros: se reserva para medir cómo funciona ante fechas posteriores. Esta separación es la que hace que la evaluación final sea una prueba de futuro y no una parte más del entrenamiento.


In [ ]:
# ==========================================================
# 12. ELEGIR MODELO, REENTRENAR HASTA 2023 Y EVALUAR TEST
# ==========================================================

# La tabla ya está ordenada por MAE medio y, como desempate, RMSE medio.
mejor_nombre = df_resultados_validacion.iloc[0]['Modelo']
mejor_configuracion = next(config for config in configuraciones if config[0] == mejor_nombre)
_, mejor_tipo, mejores_features = mejor_configuracion

print('Configuración elegida por TimeSeriesSplit:', mejor_nombre)
print('Variables:', mejores_features if mejores_features else 'baseline de media')
print('MAE medio de validación:', round(df_resultados_validacion.iloc[0]['MAE_MEUR'], 3), 'M€')

# Reentrenamos con todos los datos disponibles hasta 2023.
# Random Forest usa un máximo de 100.000 filas para mantener un tiempo de ejecución razonable.
if mejor_tipo == 'rf':
    df_train_modelo_final = df_train_final.sample(
        n=min(RF_FINAL_MAX_ROWS, len(df_train_final)),
        random_state=42
    ).copy()
else:
    df_train_modelo_final = df_train_final.copy()

print(
    f'Train final utilizado: {len(df_train_modelo_final):,} '
    f'de {len(df_train_final):,} filas disponibles.'
)

# Creamos el pipeline final y preparamos sus entradas.
mejor_modelo = crear_modelo(mejor_tipo, mejores_features)

if mejor_tipo == 'dummy':
    X_train_final = np.zeros((len(df_train_modelo_final), 1))
    X_test_final = np.zeros((len(df_test), 1))
else:
    X_train_final = df_train_modelo_final[mejores_features].copy()
    X_test_final = df_test[mejores_features].copy()
    columnas_num_final = [
        c for c in mejores_features
        if c in features_numericas or c == 'valuation_year'
    ]
    for conjunto in [X_train_final, X_test_final]:
        conjunto[columnas_num_final] = conjunto[columnas_num_final].apply(
            pd.to_numeric, errors='coerce'
        )

# El Pipeline ajusta el preprocesamiento y el modelo usando únicamente Train final.
print('Entrenando el modelo definitivo con datos hasta 2023...')
mejor_modelo.fit(X_train_final, df_train_modelo_final[target_col])

# Test se utiliza ahora solo para medir el rendimiento futuro.
print('Evaluando una vez con Test (2024 en adelante)...')
pred_test = mejor_modelo.predict(X_test_final)
metricas_test = calcular_metricas(df_test[target_col], pred_test)

df_resultado_test = pd.DataFrame([{
    'Modelo_seleccionado': mejor_nombre,
    'Estrategia_validacion': 'TimeSeriesSplit',
    'MAE_validacion_medio_MEUR': float(df_resultados_validacion.iloc[0]['MAE_MEUR']),
    'MAE_MEUR': metricas_test['MAE_MEUR'],
    'RMSE_MEUR': metricas_test['RMSE_MEUR'],
    'R2': metricas_test['R2'],
    'Test_desde': 2024
}])

print('\nRESULTADO FINAL EN TEST (2024 EN ADELANTE)')
print(df_resultado_test.round(3).to_string(index=False))

# Registramos el modelo final y el resumen de validación en MLflow, si está disponible.
final_run_id = None
final_model_logged = False

if MLFLOW_AVAILABLE:
    with mlflow.start_run(run_name=f'FINAL - {mejor_nombre}'):
        mlflow.log_params({
            'estrategia_validacion': 'TimeSeriesSplit',
            'validation_n_splits': N_SPLITS_CV,
            'validation_test_size_years': TEST_SIZE_CV,
            'modelo_seleccionado': mejor_nombre,
            'tipo_modelo': mejor_tipo,
            'features': ', '.join(mejores_features) if mejores_features else 'baseline media',
            'train_final_hasta': 2023,
            'test_desde': 2024,
            'target': target_col,
            'train_final_rows': len(df_train_modelo_final),
            'train_final_total_rows': len(df_train_final),
            'rf_max_rows': RF_FINAL_MAX_ROWS
        })
        mlflow.log_metrics({
            'validation_MAE_MEUR': float(df_resultados_validacion.iloc[0]['MAE_MEUR']),
            'validation_MAE_std_MEUR': float(df_resultados_validacion.iloc[0]['MAE_MEUR_std']),
            'validation_RMSE_MEUR': float(df_resultados_validacion.iloc[0]['RMSE_MEUR']),
            'validation_R2': float(df_resultados_validacion.iloc[0]['R2']),
            'test_MAE_MEUR': float(metricas_test['MAE_MEUR']),
            'test_RMSE_MEUR': float(metricas_test['RMSE_MEUR']),
            'test_R2': float(metricas_test['R2'])
        })

        try:
            print('Registrando el modelo final en MLflow...')
            mlflow.sklearn.log_model(mejor_modelo, artifact_path='modelo')
            final_model_logged = True
        except Exception as error:
            print('Aviso: el modelo final no pudo guardarse en MLflow:', type(error).__name__)

        final_run_id = mlflow.active_run().info.run_id

print('Run final MLflow:', final_run_id if final_run_id else 'No disponible')
print('Modelo registrado en MLflow:', final_model_logged)

### Interpretación del resultado final en Test

En las valoraciones de 2024 en adelante, el modelo seleccionado obtiene:

| Métrica | Resultado | Lectura sencilla |
|---|---:|---|
| **MAE** | **3,859 M€** | De media, la distancia absoluta entre valor real y predicción es de unos 3,859 millones de euros por valoración. |
| **RMSE** | **9,146 M€** | Es bastante mayor que el MAE, señal de que hay algunos errores grandes que pesan mucho en esta métrica. |
| **R²** | **0,393** | El modelo explica parte de la variación de los valores, pero queda bastante variabilidad sin explicar. No significa un 39,3 % de predicciones correctas. |

El MAE de Test (**3,859 M€**) es superior al MAE medio de Validation (**2,670 M€**). En otras palabras, el modelo lo hizo peor al pasar a años posteriores que en los tres años usados para compararlo. Puede deberse a cambios temporales del mercado, a la distribución de jugadores y valores en esos años o a factores que no están en las variables (contratos, lesiones, contexto deportivo, etc.). Como los valores son nominales y no se corrigen por inflación, ese aspecto también puede influir.

La lectura razonable es que Random Forest ofrece una línea base mejor que Dummy y Ridge en la comparación realizada, pero el resultado de Test muestra margen de mejora. No debemos presentarlo como una predicción exacta ni como un modelo terminado para uso profesional sin más validaciones.


## 13. Analizar errores del Test

Calculamos el error absoluto por año y mostramos valores reales frente a predicciones. El gráfico usa logaritmos solo para visualizar mejor el rango; las métricas se calcularon con las predicciones originales, sin recortarlas.

In [ ]:
# ==========================================================
# 13. ANÁLISIS DE ERRORES DEL TEST
# ==========================================================

# Tabla de predicciones y errores en euros
df_predicciones_test = df_test[
    ['valuation_id', 'player_id', 'valuation_date', 'valuation_year', 'market_value_in_eur']
].copy()
df_predicciones_test = df_predicciones_test.rename(columns={'market_value_in_eur': 'valor_real_eur'})
df_predicciones_test['prediccion_eur'] = pred_test
df_predicciones_test['error_eur'] = df_predicciones_test['prediccion_eur'] - df_predicciones_test['valor_real_eur']
df_predicciones_test['error_abs_eur'] = df_predicciones_test['error_eur'].abs()

# Resumen por año de Test
df_error_por_anio = (
    df_predicciones_test.groupby('valuation_year')
    .agg(
        valoraciones=('player_id', 'size'),
        MAE_eur=('error_abs_eur', 'mean'),
        mediana_real_eur=('valor_real_eur', 'median'),
        mediana_predicha_eur=('prediccion_eur', 'median')
    )
    .reset_index()
)
df_error_por_anio['MAE_MEUR'] = df_error_por_anio['MAE_eur'] / 1_000_000
print(df_error_por_anio.round(3).to_string(index=False))

# MAE anual: permite observar si el error aumenta en años más recientes
plt.figure(figsize=(9, 4))
plt.plot(df_error_por_anio['valuation_year'], df_error_por_anio['MAE_MEUR'], marker='o')
plt.title('MAE por año — Test final')
plt.xlabel('Año de valoración')
plt.ylabel('MAE (millones de euros)')
plt.grid(True)
plt.tight_layout()
plt.show()

# Real y predicho en log10, únicamente para el gráfico
real_log = np.log10(np.maximum(df_predicciones_test['valor_real_eur'].to_numpy(), 0) + 1)
pred_log = np.log10(np.maximum(df_predicciones_test['prediccion_eur'].to_numpy(), 0) + 1)
plt.figure(figsize=(6, 6))
plt.scatter(real_log, pred_log, alpha=0.15, s=8)
limite = max(real_log.max(), pred_log.max())
plt.plot([0, limite], [0, limite], linestyle='--')
plt.title('Valor real vs predicho (log10) — Test')
plt.xlabel('log10(valor real en € + 1)')
plt.ylabel('log10(predicción en € + 1)')
plt.grid(True)
plt.tight_layout()
plt.show()

print('Nota: predicciones negativas se muestran como cero solo en este gráfico logarítmico; no se modifican para calcular métricas.')

### Cómo interpretar los errores por año y los gráficos

El MAE anual del Test es **3,796 M€ en 2024**, **3,776 M€ en 2025** y **4,146 M€ en 2026**. En los datos guardados, el Test termina el **11 de junio de 2026**; por tanto, 2026 solo contiene una parte del año. No debemos comparar ese resultado como si tuviera doce meses de valoraciones, porque la composición y el número de registros pueden ser distintos.

Las medianas predichas que aparecen en la tabla también son superiores a las medianas reales en los tres años: aproximadamente 1,941 frente a 1,200 M€ en 2024; 1,792 frente a 1,200 M€ en 2025; y 1,957 frente a 1,500 M€ en 2026. Esto sugiere que, en el centro de la distribución, las predicciones tienden a quedar por encima del valor real. **No significa que el modelo sobrestime a todos los jugadores**: es una comparación de medianas y habría que revisar errores individuales para saber a quién sobrestima o subestima.

En el gráfico de valor real frente a predicho se utiliza `log10` únicamente para que se vean mejor los valores bajos y altos en el mismo dibujo. La línea diagonal representa el caso ideal: real y predicción iguales. Cuanto más se alejan los puntos de esa línea, mayor es el error.

El código muestra como cero las predicciones negativas **solo para poder dibujarlas en escala logarítmica**. No modifica las predicciones originales ni las métricas. Por tanto, si el modelo genera alguna predicción negativa, esa limitación sigue existiendo en el resultado numérico; este notebook no aplica un recorte a cero en el modelo.


## 14. Consultar ejecuciones de MLflow

Si MLflow está disponible, mostramos las ejecuciones finales registradas. Las ejecuciones de cada pliegue también quedan guardadas, con sus métricas individuales.

No iniciamos ningún servidor desde este notebook. Si MLflow no está disponible, los resultados siguen guardados en los CSV de Drive.

In [ ]:
# ==========================================================
# 14. CONSULTAR EXPERIMENTOS DE MLFLOW
# ==========================================================

if MLFLOW_AVAILABLE:
    experimento = mlflow.get_experiment_by_name('OFFBALL_Regresion_Valor_Mercado')

    if experimento is not None:
        df_runs_mlflow = mlflow.search_runs(
            experiment_ids=[experimento.experiment_id]
        )

        # Mostramos primero los modelos finales; los pliegues CV quedan registrados por separado.
        if 'tags.mlflow.runName' in df_runs_mlflow.columns:
            df_runs_finales = df_runs_mlflow[
                df_runs_mlflow['tags.mlflow.runName'].str.startswith('FINAL -', na=False)
            ].copy()
        else:
            df_runs_finales = df_runs_mlflow.copy()

        columnas_runs = [
            c for c in [
                'run_id',
                'tags.mlflow.runName',
                'params.estrategia_validacion',
                'metrics.validation_MAE_MEUR',
                'metrics.test_MAE_MEUR',
                'metrics.test_RMSE_MEUR',
                'metrics.test_R2'
            ]
            if c in df_runs_finales.columns
        ]

        if len(df_runs_finales):
            print(df_runs_finales[columnas_runs].head(10).to_string(index=False))
        else:
            print('Todavía no hay ejecuciones finales registradas.')
    else:
        print('El experimento todavía no existe.')
else:
    print('MLflow no está disponible. Los resultados se revisan en los CSV de Drive.')

### Qué significa que MLflow no esté disponible en esta ejecución

La salida indica que MLflow no está disponible. El notebook no lo instala ni inicia un servidor; por tanto, en esta ejecución **no se han registrado runs ni el modelo en MLflow**. Eso no impide completar la regresión: las métricas y el modelo se guardan en Google Drive como archivos locales.

Es importante distinguir ambas cosas al explicarlo: el seguimiento con MLflow queda como capacidad opcional, pero el artefacto local de `joblib` y los CSV sí se generan. No debemos afirmar que el modelo está registrado en MLflow si la salida dice que no está disponible.


## 15. Probar la carga del modelo desde MLflow

Si el modelo final se registró correctamente, lo cargamos desde su `run_id` y hacemos una predicción de prueba. Esto verifica la forma de recuperar el artefacto desde la librería MLflow; no inicia el servidor ni entrena de nuevo.


In [ ]:
# ==========================================================
# 15. CARGAR EL MODELO FINAL DESDE MLFLOW
# ==========================================================

# Esta comprobación carga el artefacto desde MLflow y no es necesaria en cada ejecución.
# Activarla (True) una vez cuando queramos verificar que se puede recuperar el modelo.
COMPROBAR_CARGA_MLFLOW = False

model_uri_mlflow = f'runs:/{final_run_id}/modelo' if (final_run_id and final_model_logged) else None
if COMPROBAR_CARGA_MLFLOW and MLFLOW_AVAILABLE and model_uri_mlflow:
    try:
        print('Cargando artefacto final desde MLflow...')
        modelo_cargado_mlflow = mlflow.sklearn.load_model(model_uri_mlflow)
        predicciones_demo = modelo_cargado_mlflow.predict(X_test_final[:5])
        print('Modelo recuperado desde MLflow:', model_uri_mlflow)
        print('Predicciones de prueba (euros):', np.round(predicciones_demo, 2))
    except Exception as error:
        print('No se pudo cargar el modelo desde MLflow:', type(error).__name__)
else:
    print('Carga desde MLflow omitida (opcional). El modelo final se guardó localmente con joblib.')


### Por qué la carga desde MLflow es opcional

La variable `COMPROBAR_CARGA_MLFLOW` está en `False`, así que esta comprobación se omite. Si MLflow estuviera configurado y el modelo se hubiera registrado, permitiría comprobar que se puede recuperar desde su identificador y hacer unas predicciones de prueba.

Esto no es un segundo entrenamiento. Para el despliegue previsto, FastAPI cargará el fichero local `mejor_modelo.joblib`; la API debe reutilizar el modelo ya entrenado, no volver a entrenarlo cada vez que arranca.


## 16. Guardar dataset, métricas y modelo

Guardamos el dataset preparado, auditorías, resultados medios de `TimeSeriesSplit`, resultados de cada pliegue, predicciones de Test, el modelo final y un JSON con su configuración.

Los archivos con el mismo nombre se actualizan al volver a ejecutar el notebook. El modelo local se guarda aunque MLflow no esté disponible.

### Por qué guardamos varios archivos y no solo el modelo

Cada salida tiene una finalidad diferente:

- Los CSV de dataset, auditoría y cobertura permiten revisar qué datos entraron y qué filtros se aplicaron.
- Los CSV de validación, pliegues y Test permiten comprobar cómo rindieron las configuraciones y dónde cometió errores el modelo.
- `mejor_modelo_info.json` guarda la ficha del modelo: algoritmo elegido, variables, periodos, estrategia y métricas.
- `mejor_modelo.joblib` guarda el **pipeline entrenado completo**, incluido el preprocesamiento y el modelo. Así, al predecir con nuevos jugadores, se aplican las mismas transformaciones que durante el entrenamiento.

Esta separación facilita la auditoría del experimento y permite que el compañero que despliegue la API use el artefacto final sin reconstruir manualmente el preprocesamiento.


In [ ]:
# ==========================================================
# 16. GUARDAR ARTEFACTOS DE LA REGRESIÓN
# ==========================================================

import joblib
import json

# Definimos las rutas de salida en Google Drive.
ruta_df2 = OUTPUT_DIR / 'df2_regression.csv'
ruta_auditoria = OUTPUT_DIR / 'auditoria_filtros_regresion.csv'
ruta_cobertura = OUTPUT_DIR / 'cobertura_estadisticas.csv'
ruta_metricas_validacion = OUTPUT_DIR / 'metricas_validacion_regresion.csv'
ruta_detalle_folds = OUTPUT_DIR / 'detalle_folds_timeseriessplit.csv'
ruta_metricas_test = OUTPUT_DIR / 'metricas_test_modelo_seleccionado.csv'
ruta_predicciones_test = OUTPUT_DIR / 'predicciones_test_modelo_seleccionado.csv'
ruta_error_anual = OUTPUT_DIR / 'error_test_por_anio.csv'
ruta_diccionario = OUTPUT_DIR / 'diccionario_variables_regresion.csv'
ruta_modelo = MODELS_DIR / 'mejor_modelo.joblib'
ruta_info_modelo = MODELS_DIR / 'mejor_modelo_info.json'

# Guardamos los datos y resultados para poder revisarlos fuera del notebook.
df2_regression.to_csv(ruta_df2, index=False)
df_auditoria_filtros.to_csv(ruta_auditoria, index=False)
df_cobertura_anual.to_csv(ruta_cobertura, index=False)
df_resultados_validacion.to_csv(ruta_metricas_validacion, index=False)
df_resultados_folds.to_csv(ruta_detalle_folds, index=False)
df_resultado_test.to_csv(ruta_metricas_test, index=False)
df_predicciones_test.to_csv(ruta_predicciones_test, index=False)
df_error_por_anio.to_csv(ruta_error_anual, index=False)

# Si hay un run final de MLflow, añadimos los CSV como artefactos de apoyo.
if MLFLOW_AVAILABLE and final_run_id:
    try:
        with mlflow.start_run(run_id=final_run_id):
            for archivo_artifacto in [
                ruta_metricas_validacion,
                ruta_detalle_folds,
                ruta_metricas_test,
                ruta_predicciones_test,
                ruta_error_anual
            ]:
                mlflow.log_artifact(str(archivo_artifacto))
        print('Tablas de apoyo registradas como artefactos en MLflow.')
    except Exception as error:
        print('Aviso: no se pudieron registrar las tablas en MLflow:', type(error).__name__)

# Guardamos un diccionario sencillo de las variables principales.
diccionario_variables = pd.DataFrame([
    ('valuation_id', 'Identificador creado en este notebook para cada valoración'),
    ('player_id', 'Identificador del jugador en la fuente'),
    ('valuation_date', 'Fecha de la valoración de mercado'),
    ('valuation_year', 'Año de la valoración'),
    ('age', 'Edad completa del jugador en la fecha de valoración'),
    ('position', 'Posición general del jugador'),
    ('sub_position', 'Subposición específica del jugador'),
    ('competiciones_con_stats', 'Número de competiciones con estadísticas disponibles'),
    ('latest_stats_end', 'Fecha final de la última temporada utilizada'),
    ('partidos', 'Partidos agregados de las temporadas seleccionadas'),
    ('minutos', 'Minutos agregados de las temporadas seleccionadas'),
    ('goles', 'Goles agregados de las temporadas seleccionadas'),
    ('asistencias', 'Asistencias agregadas de las temporadas seleccionadas'),
    ('goles_por90', 'Goles por 90 calculados sobre los totales agregados'),
    ('asistencias_por90', 'Asistencias por 90 calculadas sobre los totales agregados'),
    ('market_value_in_eur', 'Objetivo: valor de mercado en euros nominales')
], columns=['Variable', 'Descripción'])
diccionario_variables.to_csv(ruta_diccionario, index=False)

# Guardamos el modelo entrenado: este es el archivo que luego podrá cargar la API.
joblib.dump(mejor_modelo, ruta_modelo)

# Guardamos su ficha técnica: estrategia, variables, periodos y resultados.
# validation_MAE_MEUR representa la media del MAE de los pliegues de TimeSeriesSplit.
model_uri_mlflow = (
    f'runs:/{final_run_id}/modelo'
    if (final_run_id and final_model_logged)
    else None
)

info_modelo = {
    'modelo_seleccionado': mejor_nombre,
    'tipo': mejor_tipo,
    'features': mejores_features,
    'target': target_col,
    'validation_strategy': 'TimeSeriesSplit',
    'validation_split_unit': 'valuation_year',
    'validation_n_splits': int(N_SPLITS_CV),
    'validation_test_size_years': int(TEST_SIZE_CV),
    'validation_folds': resumen_folds_cv,
    'validation_MAE_MEUR': float(df_resultados_validacion.iloc[0]['MAE_MEUR']),
    'validation_MAE_std_MEUR': float(df_resultados_validacion.iloc[0]['MAE_MEUR_std']),
    'validation_RMSE_MEUR': float(df_resultados_validacion.iloc[0]['RMSE_MEUR']),
    'validation_RMSE_std_MEUR': float(df_resultados_validacion.iloc[0]['RMSE_MEUR_std']),
    'validation_R2': float(df_resultados_validacion.iloc[0]['R2']),
    'train_final_hasta': 2023,
    'test_desde': 2024,
    'train_final_rows': int(len(df_train_modelo_final)),
    'train_final_total_rows': int(len(df_train_final)),
    'test_rows': int(len(df_test)),
    'test_MAE_MEUR': float(metricas_test['MAE_MEUR']),
    'test_RMSE_MEUR': float(metricas_test['RMSE_MEUR']),
    'test_R2': float(metricas_test['R2']),
    'mlflow_tracking_uri': mlflow.get_tracking_uri() if MLFLOW_AVAILABLE else None,
    'mlflow_final_run_id': final_run_id,
    'mlflow_model_uri': model_uri_mlflow
}

with open(ruta_info_modelo, 'w', encoding='utf-8') as archivo_json:
    json.dump(info_modelo, archivo_json, ensure_ascii=False, indent=2)

# Confirmamos las rutas de todos los archivos generados o actualizados.
print('Artefactos guardados:')
for ruta in [
    ruta_df2, ruta_auditoria, ruta_cobertura,
    ruta_metricas_validacion, ruta_detalle_folds,
    ruta_metricas_test, ruta_predicciones_test, ruta_error_anual,
    ruta_diccionario, ruta_modelo, ruta_info_modelo
]:
    print('-', ruta)

### Qué queda disponible al terminar esta ejecución

La salida confirma que los CSV, el modelo `mejor_modelo.joblib` y su ficha `mejor_modelo_info.json` se han guardado en la carpeta de regresión de Drive. Los archivos de resultados permiten revisar el experimento sin tener que volver a ejecutar todos los cálculos.

Los nombres de salida son fijos: al ejecutar de nuevo el notebook, los archivos con esos mismos nombres se **sobrescriben** con los nuevos resultados. Si necesitamos conservar una versión anterior para comparar, debemos copiarla o cambiarle el nombre antes de volver a ejecutar.


## 17. Qué queda para la siguiente fase

- Revisar los resultados reales de los tres pliegues y del Test final.
- Compartir con el compañero de clustering los archivos `mejor_modelo.joblib` y `mejor_modelo_info.json`. Son la **línea base de regresión sin GMM**.
- Cuando el compañero entregue `player_id` y las etiquetas/probabilidades GMM, unirlas sin duplicar valoraciones ni utilizar información futura.
- Comparar el modelo base con otro modelo que incorpore GMM, usando los mismos pliegues temporales y el mismo Test final. Guardar el modelo con GMM con otro nombre para no sobrescribir la línea base.
- Para el despliegue, FastAPI cargará el `.joblib` ya entrenado; la API no debe reentrenar el modelo al arrancar.

**Limitaciones a documentar:** valores nominales sin ajuste de inflación; estadísticas generales y no específicas por posición; jugadores pueden aparecer en Train y Test en fechas distintas porque evaluamos su evolución futura; y las fechas no permiten ordenar eventos del mismo día.

## Resumen final: qué hemos decidido y qué podemos afirmar

1. **Qué predecimos:** el valor de mercado en euros de cada valoración de un jugador.
2. **Con qué información:** edad, minutos, goles por 90, asistencias por 90, posición y subposición. El año se probó, pero no formó parte de la configuración ganadora.
3. **Cómo evitamos mirar al futuro:** usamos estadísticas de temporadas terminadas antes de cada valoración y validamos por años, dejando 2024 en adelante para el Test.
4. **Qué modelo ganó:** Random Forest sin año, porque logró el MAE medio más bajo en los tres años de Validation.
5. **Qué resultado obtuvo:** MAE de Test de 3,859 M€, RMSE de 9,146 M€ y R² de 0,393. Es una línea base útil, con errores todavía importantes.

### Limitaciones que debemos explicar con claridad

- Los valores son nominales: no se ha corregido la inflación ni la evolución general del mercado.
- Las variables de rendimiento son generales. No incluyen, por ejemplo, estadísticas defensivas específicas, lesiones, duración del contrato o contexto del equipo.
- Se utiliza la posición disponible en `players.csv`; no se reconstruye un historial de posiciones para cada fecha.
- No se ha exigido un mínimo de 450 minutos. Esto conserva más valoraciones, pero algunos ratios por 90 pueden ser poco estables si el jugador tuvo pocos minutos.
- Un jugador puede aparecer en Train y Test en fechas diferentes. La prueba mide predicciones futuras de valoraciones, incluso de jugadores ya conocidos; **no mide el rendimiento con futbolistas completamente nuevos**.
- El modelo final se entrenó con una muestra de 100.000 de las 235.103 filas disponibles hasta 2023 para reducir el coste de cálculo.
- El Test de 2026 llega solo hasta el 11 de junio de 2026, por lo que ese año está incompleto.
- GMM todavía no está integrado en esta línea base y MLflow no estuvo disponible en esta ejecución.

La siguiente comparación con clustering debe mantener los mismos periodos de Train, Validation y Test. El GMM se debe ajustar solo con Train y después aplicarse a Validation y Test, para evitar que la información futura influya en las etiquetas o probabilidades de los jugadores.
